# 205 — Trace Repository & Trace Tools (Neo4j)
Store investigation traces as a subgraph connected to existing business nodes.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import uuid
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tools.trace_tools import TraceTools
from src.domain.models import EventType, InvestigationTrace, TraceEvent

settings = get_neo4j_settings()
neo4j = Neo4jRepository(**vars(settings))
neo4j.test_connection()

repo = TraceRepository(neo4j)
tools = TraceTools(repo)
print("Connected")

Connected


## Create and save a trace

In [3]:
trace = InvestigationTrace(
    request_id=str(uuid.uuid4()),
    entity_name="TELEFONICA O2 HOLDINGS LIMITED",
    user_id="analyst-001",
    mode="interactive",
)

trace_id = repo.save_trace(trace)
print(f"Saved trace: {trace_id}")

Saved trace: 31fd157c-44b0-4c15-bf6e-21349f363564


## Append events — with and without entity refs

In [4]:
# Event 1 — plan created, no entity ref
repo.append_event(
    trace_id,
    TraceEvent(
        event_type=EventType.PLAN_CREATED,
        message="Plan created with 3 steps.",
        payload={"step_count": 3},
    ),
)

# Event 2 — tool called, linked to Company node
repo.append_event(
    trace_id,
    TraceEvent(
        event_type=EventType.TOOL_CALLED,
        step_id="step_1",
        message="Calling ownership_complexity_check.",
        payload={
            "tool_name": "ownership_complexity_check",
            "input_summary": "company_name=TELEFONICA O2 HOLDINGS LIMITED",
        },
    ),
    entity_refs=[
        {"label": "Company", "name": "TELEFONICA O2 HOLDINGS LIMITED"},
    ],
)

# Event 3 — tool returned, linked to Company + its owner
repo.append_event(
    trace_id,
    TraceEvent(
        event_type=EventType.TOOL_RETURNED,
        step_id="step_1",
        message="ownership_complexity_check returned MEDIUM risk.",
        payload={
            "tool_name": "ownership_complexity_check",
            "output_summary": "risk_level=MEDIUM, max_depth=1, ubo_count=0",
            "decision": "proceed to address check",
            "why": "corporate chain only — no individual UBOs found",
        },
    ),
    entity_refs=[
        {"label": "Company", "name": "TELEFONICA O2 HOLDINGS LIMITED"},
        {"label": "Company", "name": "Telefonica S A"},
    ],
)

# Event 4 — failed step
repo.append_event(
    trace_id,
    TraceEvent(
        event_type=EventType.STEP_FAILED,
        step_id="step_2",
        message="address_risk_check failed: no address node found.",
        payload={"tool_name": "address_risk_check"},
    ),
    entity_refs=[
        {"label": "Company", "name": "TELEFONICA O2 HOLDINGS LIMITED"},
    ],
)

print("Events appended")

Events appended


## Finalize trace

In [5]:
repo.finalize_trace(
    trace_id,
    final_summary=(
        "Telefonica O2 Holdings Limited is wholly owned by Telefonica S.A. "
        "(75-100%). Address check failed. Ownership complexity: MEDIUM."
    ),
)
print("Trace finalized")

Trace finalized


## list_recent_traces

In [6]:
r = tools.list_recent_traces(limit=10)
print(f"[{'✓' if r.success else '✗'}] {r.tool_name}  ({r.duration_ms} ms)")
print(f"    {r.summary}")
if r.data:
    display(pd.DataFrame(r.data))

[✓] list_recent_traces  (17.8 ms)
    Found 1 recent trace(s).


,trace_id,query,user_id,mode,started_at,ended_at,event_count
0,31fd157c-44b0-4c15-bf6e-21349f363564,TELEFONICA O2 HOLDINGS LIMITED,analyst-001,interactive,2026-03-22T02:46:53.770628+00:00,2026-03-22T02:46:56.629966+00:00,4


## find_traces_by_entity

In [7]:
r = tools.find_traces_by_entity("TELEFONICA O2 HOLDINGS LIMITED")
print(f"[{'✓' if r.success else '✗'}] {r.tool_name}  ({r.duration_ms} ms)")
print(f"    {r.summary}")
if r.data:
    display(pd.DataFrame(r.data))

[✓] find_traces_by_entity  (17.3 ms)
    Found 1 trace(s) connected to 'TELEFONICA O2 HOLDINGS LIMITED'.


,trace_id,query,user_id,started_at,ended_at
0,31fd157c-44b0-4c15-bf6e-21349f363564,TELEFONICA O2 HOLDINGS LIMITED,analyst-001,2026-03-22T02:46:53.770628+00:00,2026-03-22T02:46:56.629966+00:00


## retrieve_trace

In [8]:
r = tools.retrieve_trace(trace_id)
print(f"[{'✓' if r.success else '✗'}] {r.tool_name}  ({r.duration_ms} ms)")
print(f'    {r.summary}')
if r.data:
    print(f"\nfinal_summary: {r.data['final_summary']}")
    print('\nEvents:')
    events_df = pd.DataFrame(r.data["events"])
    display(events_df[["event_number", "event_type", "tool_name", "input_summary", "output_summary", "decision", "about"]])


[✓] retrieve_trace  (45.4 ms)
    Trace '31fd157c-44b0-4c15-bf6e-21349f363564' for 'TELEFONICA O2 HOLDINGS LIMITED': 4 event(s).

final_summary: Telefonica O2 Holdings Limited is wholly owned by Telefonica S.A. (75-100%). Address check failed. Ownership complexity: MEDIUM.

Events:


,event_number,event_type,tool_name,input_summary,output_summary,decision,about
0,1,plan_created,,Plan created with 3 steps.,,,"[{'identifier': '', 'labels': None}]"
1,2,tool_called,ownership_complexity_check,company_name=TELEFONICA O2 HOLDINGS LIMITED,,,[{'identifier': 'TELEFONICA O2 HOLDINGS LIMITE...
2,3,tool_returned,ownership_complexity_check,ownership_complexity_check returned MEDIUM risk.,"risk_level=MEDIUM, max_depth=1, ubo_count=0",proceed to address check,[{'identifier': 'TELEFONICA O2 HOLDINGS LIMITE...
3,4,step_failed,address_risk_check,address_risk_check failed: no address node found.,,,[{'identifier': 'TELEFONICA O2 HOLDINGS LIMITE...


## retrieve_trace — miss case

In [9]:
r = tools.retrieve_trace("does-not-exist")
print(f"success: {r.success}")
print(f"error  : {r.error}")

success: False
error  : No trace found with id 'does-not-exist'.


## Verify graph in Neo4j Browser

Event 1 (`plan_created`) was appended without `entity_refs`, so no `:ABOUT`
relationship was created for it. Use `OPTIONAL MATCH` to see all events —
event 1 will appear with no connected business node:

```cypher
MATCH (t:InvestigationTrace {trace_id: '<paste trace_id>'})-[:HAS_EVENT]->(e)
OPTIONAL MATCH (e)-[:ABOUT]->(n)
RETURN t, e, n
```

To see only events linked to a business entity (events 2–4), use a required
match — this excludes event 1:

```cypher
MATCH (t:InvestigationTrace {trace_id: '<paste trace_id>'})-[:HAS_EVENT]->(e)-[:ABOUT]->(n)
RETURN t, e, n
```

Each event (except event 1) is also linked to its predecessor via `:NEXT_EVENT`.
To walk the event chain as consecutive pairs:

```cypher
MATCH (t:InvestigationTrace {trace_id: '<paste trace_id>'})-[:HAS_EVENT]->(e1)-[:NEXT_EVENT]->(e2)
RETURN e1.event_number, e1.event_type, e2.event_number, e2.event_type
```

## Cleanup — delete this notebook's trace from Neo4j

In [10]:
# Delete the InvestigationTrace node, its TraceEvent nodes, and all :ABOUT/:HAS_EVENT relationships.
# Business nodes (Company, Address, etc.) are NOT deleted.
neo4j.run_query(
    """
    MATCH (t:InvestigationTrace {trace_id: $trace_id})
    OPTIONAL MATCH (t)-[:HAS_EVENT]->(e:TraceEvent)
    DETACH DELETE t, e
    """,
    {"trace_id": trace_id},
)
print(f"Deleted trace {trace_id} and its events")


Deleted trace 31fd157c-44b0-4c15-bf6e-21349f363564 and its events


In [11]:
neo4j.close()
print("Driver closed")

Driver closed
